# Prepare shared autocorrelated additive structural causal process data

This notebook creates or verifies the frozen `autocorrelatedSCM` bundles used by
all eight method notebooks in this directory. Data generation and acquisition
are kept separate from model fitting so every method reads byte-identical
observations and the same declared ground truth.


The process follows the simulation family used by Debeire et al. (2024):

\[
X_t^j = a_j X_{t-1}^j
  + \sum_i c_i f_i(X_{t-\tau_i}^i) + \eta_t^j.
\]

The identity link is \(f(x)=x\). The second link option is
\(f(x)=x+5x^2\exp(-x^2/20)\). Cross-link coefficients are sampled from
\(\{\pm0.1,\ldots,\pm0.5\}\), lags range from 1 to 5, innovation scales
range from 0.5 to 2, and each accepted process passes the authors'
stationarity check. The study varies variable count, sample count,
autocorrelation, link-function set, and innovation family one factor at a
time around a reference cell, with 50 seeds per cell. A cross-link-free cell
measures false positives when the cross-variable graph is empty.

The paper's broad simulation includes 30% contemporaneous cross-links. This
matched comparison arm sets that fraction to zero because all eight methods
can be scored on a shared positive-lag graph, whereas they do not expose a
common contemporaneous estimand. This is a declared adaptation, not a claim of
literal reproduction. The notebook calls the official generator from its
pinned authors' checkout and records every accepted seed and rejected-model
count.

In [ ]:
from __future__ import annotations

import hashlib
import importlib.util
import json
import math
import os
import platform
import subprocess
import sys
import tempfile
import types
import urllib.request
import zipfile
import pickle
from pathlib import Path

EXPECTED_PYTHON = (3, 14)
if sys.version_info[:2] != EXPECTED_PYTHON:
    raise RuntimeError(
        "This notebook requires CPython 3.14; "
        f"found {platform.python_version()}."
    )

import numpy as np
import pandas as pd
import scipy.special

PROJECT_ROOT = next(
    (
        path
        for path in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
        if (path / "src" / "fastCGC" / "__init__.py").exists()
    ),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Could not locate the fastC-GC project root.")
SOURCE_ROOT = PROJECT_ROOT / "src"
if str(SOURCE_ROOT) not in sys.path:
    sys.path.insert(0, str(SOURCE_ROOT))

from fastCGC.experiments.upstream import verify_upstream_revision

In [ ]:
def series_digest(series):
    return hashlib.sha256(
        np.ascontiguousarray(series, dtype=np.float64).tobytes()
    ).hexdigest()


def save_bundle(
    path,
    *,
    series,
    metadata,
    aggregate_truth=None,
    lag_truth=None,
    reference_graph=None,
    extra_arrays=None,
):
    graph_arrays = (aggregate_truth, lag_truth, reference_graph)
    if sum(value is not None for value in graph_arrays) > 1:
        raise ValueError("Provide at most one truth or reference graph.")
    path.parent.mkdir(parents=True, exist_ok=True)
    arrays = {"series": np.asarray(series, dtype=float)}
    if lag_truth is not None:
        arrays["lag_truth"] = np.asarray(lag_truth, dtype=bool)
    elif aggregate_truth is not None:
        arrays["aggregate_truth"] = np.asarray(
            aggregate_truth,
            dtype=bool,
        )
    elif reference_graph is not None:
        arrays["reference_graph"] = np.asarray(
            reference_graph,
            dtype=bool,
        )
    for name, value in (extra_arrays or {}).items():
        if name in arrays:
            raise ValueError(f"Duplicate bundle array name: {name}.")
        arrays[name] = np.asarray(value)
    digest = series_digest(arrays["series"])
    metadata = {**metadata, "data_digest": digest}
    metadata_path = path.with_suffix(".json")
    if path.exists() or metadata_path.exists():
        if not path.is_file() or not metadata_path.is_file():
            raise FileNotFoundError(f"Incomplete dataset bundle at {path}.")
        recorded = json.loads(metadata_path.read_text(encoding="utf-8"))
        with np.load(path, allow_pickle=False) as stored:
            saved = {name: np.asarray(stored[name]) for name in stored.files}
        arrays_match = set(saved) == set(arrays) and all(
            np.array_equal(saved[name], value)
            for name, value in arrays.items()
        )
        if not arrays_match or recorded != metadata:
            raise ValueError(f"Existing bundle does not match request: {path}.")
        return False

    with tempfile.NamedTemporaryFile(
        mode="wb", dir=path.parent, prefix=f".{path.name}.", delete=False
    ) as handle:
        np.savez_compressed(handle, **arrays)
        temporary_data = Path(handle.name)
    os.replace(temporary_data, path)

    with tempfile.NamedTemporaryFile(
        mode="w",
        encoding="utf-8",
        dir=path.parent,
        prefix=f".{metadata_path.name}.",
        delete=False,
    ) as handle:
        json.dump(metadata, handle, indent=2, sort_keys=True)
        handle.write("\n")
        temporary_metadata = Path(handle.name)
    os.replace(temporary_metadata, metadata_path)
    return True


def file_digest(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def download_verified(url, path, expected_sha256):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    if path.is_file():
        observed = file_digest(path)
        if observed != expected_sha256:
            raise ValueError(
                f"File digest mismatch for {path}: {observed}."
            )
        return False

    with tempfile.NamedTemporaryFile(
        mode="wb",
        dir=path.parent,
        prefix=f".{path.name}.",
        delete=False,
    ) as handle:
        temporary = Path(handle.name)
        with urllib.request.urlopen(url, timeout=120) as response:
            while chunk := response.read(1024 * 1024):
                handle.write(chunk)
    try:
        observed = file_digest(temporary)
        if observed != expected_sha256:
            raise ValueError(
                f"Downloaded file digest mismatch: {observed}."
            )
        os.replace(temporary, path)
    finally:
        if temporary.exists():
            temporary.unlink()
    return True

In [ ]:
DATASET_KEY = "autocorrelatedSCM"
DATASET_DIR = (
    PROJECT_ROOT / "data" / "shared" / "causal-comparison"
    / DATASET_KEY / "datasets"
)
GENERATOR_ROOT = Path(
    os.environ.get(
        "BAGGED_PCMCI_DATA_ROOT",
        PROJECT_ROOT
        / "external"
        / "debeire24clear_Bagged_TimeSeries_Causality",
    )
).expanduser()
UPSTREAM_REVISION = verify_upstream_revision(
    GENERATOR_ROOT,
    expected_url="https://github.com/EyringMLClimateGroup/debeire24clear_Bagged_TimeSeries_Causality.git",
    expected_commit="8e16fb89b8f3b4db47ff57f2bd09dcd5c9f7e4c9",
)
generator_path = GENERATOR_ROOT / "generate_data_mod.py"
spec = importlib.util.spec_from_file_location(
    "debeire24_generate_data_mod",
    generator_path,
)
if spec is None or spec.loader is None:
    raise ImportError(f"Could not load the author generator at {generator_path}.")
author_generator = importlib.util.module_from_spec(spec)
temporary_networkx_module = False
if importlib.util.find_spec("networkx") is None:
    generator_source = generator_path.read_text(encoding="utf-8")
    if "nx." in generator_source:
        raise ModuleNotFoundError(
            "The pinned generator now uses networkx; install the author "
            "repository's dependencies before running this notebook."
        )
    # The pinned file imports networkx but never references its `nx` alias.
    # Supplying an empty module avoids adding an unused project dependency
    # while leaving every executed generator definition unchanged.
    sys.modules["networkx"] = types.ModuleType("networkx")
    temporary_networkx_module = True
try:
    spec.loader.exec_module(author_generator)
finally:
    if temporary_networkx_module:
        sys.modules.pop("networkx", None)


def identity_link(x):
    return x


def quadratic_taper_link(x):
    return x + 5.0 * x**2 * np.exp(-(x**2) / 20.0)


def make_noises(random_state, choices, n_variables):
    functions = []
    assigned = []
    scales = []
    weibull_shape = 2.0
    weibull_mean = scipy.special.gamma(1.0 / weibull_shape + 1.0)
    weibull_variance = (
        scipy.special.gamma(2.0 / weibull_shape + 1.0)
        - weibull_mean**2
    )
    for _ in range(n_variables):
        noise_type = str(random_state.choice(choices))
        scale = float(random_state.uniform(0.5, 2.0))
        if noise_type == "gaussian":
            def draw(size, scale=scale):
                return scale * random_state.randn(size)
        elif noise_type == "weibull":
            def draw(size, scale=scale):
                values = random_state.weibull(weibull_shape, size=size)
                return scale * (values - weibull_mean) / np.sqrt(
                    weibull_variance
                )
        else:
            raise ValueError(f"Unknown noise type: {noise_type}.")
        functions.append(draw)
        assigned.append(noise_type)
        scales.append(scale)
    return functions, assigned, scales


def lag_arrays(links, *, n_variables, fit_order):
    truth = np.zeros((fit_order, n_variables, n_variables), dtype=bool)
    coefficients = np.zeros((fit_order, n_variables, n_variables), dtype=float)
    function_codes = np.full(
        (fit_order, n_variables, n_variables),
        -1,
        dtype=np.int8,
    )
    for target, parents in links.items():
        for (source, signed_lag), coefficient, function in parents:
            if signed_lag >= 0:
                raise ValueError("The matched comparison arm must contain only lags.")
            lag_index = abs(int(signed_lag)) - 1
            truth[lag_index, source, target] = True
            coefficients[lag_index, source, target] = float(coefficient)
            function_codes[lag_index, source, target] = (
                0 if function.__name__ in {"lin", "identity_link"} else 1
            )
    return truth, coefficients, function_codes


# One-factor-at-a-time design around the paper-matched reference cell. Link
# functions and innovation families are described directly rather than being
# split into separate linear/nonlinear experiment trees.
REGIMES = (
    {"regime": "reference", "n_variables": 10, "n_samples": 500,
      "max_auto": 0.60, "functions": ("identity",),
      "noise_types": ("gaussian",)},
    {"regime": "variables-5", "n_variables": 5, "n_samples": 500,
      "max_auto": 0.60, "functions": ("identity",),
      "noise_types": ("gaussian",)},
    {"regime": "variables-20", "n_variables": 20, "n_samples": 500,
      "max_auto": 0.60, "functions": ("identity",),
      "noise_types": ("gaussian",)},
    {"regime": "samples-250", "n_variables": 10, "n_samples": 250,
      "max_auto": 0.60, "functions": ("identity",),
      "noise_types": ("gaussian",)},
    {"regime": "samples-1000", "n_variables": 10, "n_samples": 1000,
      "max_auto": 0.60, "functions": ("identity",),
      "noise_types": ("gaussian",)},
    {"regime": "autocorrelation-0.20", "n_variables": 10,
      "n_samples": 500, "max_auto": 0.20,
      "functions": ("identity",), "noise_types": ("gaussian",)},
    {"regime": "autocorrelation-0.95", "n_variables": 10,
      "n_samples": 500, "max_auto": 0.95,
      "functions": ("identity",), "noise_types": ("gaussian",)},
    {"regime": "mixed-link-functions", "n_variables": 10,
      "n_samples": 500, "max_auto": 0.60,
      "functions": ("identity", "quadratic-taper"),
      "noise_types": ("gaussian",)},
    {"regime": "mixed-innovations", "n_variables": 10,
      "n_samples": 500, "max_auto": 0.60,
      "functions": ("identity",),
      "noise_types": ("gaussian", "weibull")},
    {"regime": "cross-link-null", "n_variables": 10,
      "n_samples": 500, "max_auto": 0.60,
      "functions": ("identity",), "noise_types": ("gaussian",),
      "cross_link_factor": 0.0},
)
FIT_ORDER = 5
REPLICATES = 50
SEED_START = 30_000
MAX_STATIONARITY_ATTEMPTS = 1000
MAX_NEW_DATASETS: int | None = None

created = 0
stop_requested = False
for regime_index, regime in enumerate(REGIMES):
    n_variables = int(regime["n_variables"])
    n_samples = int(regime["n_samples"])
    n_cross_links = math.floor(
        float(regime.get("cross_link_factor", 1.5)) * n_variables
    )
    functions = [
        identity_link if name == "identity" else quadratic_taper_link
        for name in regime["functions"]
    ]
    positive_couplings = np.arange(0.1, 0.51, 0.1).tolist()
    couplings = positive_couplings + [-value for value in positive_couplings]
    auto_coefficients = np.arange(
        max(0.0, float(regime["max_auto"]) - 0.3),
        float(regime["max_auto"]) + 0.01,
        0.05,
    ).tolist()

    for replicate in range(REPLICATES):
        if MAX_NEW_DATASETS is not None and created >= MAX_NEW_DATASETS:
            stop_requested = True
            break
        requested_seed = SEED_START + regime_index * REPLICATES + replicate
        model_seed = requested_seed
        nonstationary = True
        for retry in range(MAX_STATIONARITY_ATTEMPTS):
            random_state = np.random.RandomState(model_seed)
            links = author_generator.generate_random_contemp_model(
                N=n_variables,
                L=n_cross_links,
                coupling_coeffs=couplings,
                coupling_funcs=functions,
                auto_coeffs=auto_coefficients,
                tau_max=FIT_ORDER,
                contemp_fraction=0.0,
                random_state=random_state,
            )
            noises, assigned_noises, noise_scales = make_noises(
                random_state,
                regime["noise_types"],
                n_variables,
            )
            generated, nonstationary = (
                author_generator.generate_nonlinear_contemp_timeseries(
                    links=links,
                    T=n_samples + 10_000,
                    noises=noises,
                    random_state=random_state,
                )
            )
            if not nonstationary:
                series_time_variable = np.asarray(generated[:n_samples], dtype=float)
                break
            model_seed += 10_000
        if nonstationary:
            raise RuntimeError(
                f"No stationary process found for {regime['regime']} "
                f"replicate {replicate} after {MAX_STATIONARITY_ATTEMPTS} attempts."
            )

        lag_truth, lag_coefficients, lag_function_codes = lag_arrays(
            links,
            n_variables=n_variables,
            fit_order=FIT_ORDER,
        )
        dataset_id = (
            f"acscm-{regime['regime']}-seed{requested_seed}"
        )
        was_created = save_bundle(
            DATASET_DIR / f"{dataset_id}.npz",
            series=series_time_variable.T,
            lag_truth=lag_truth,
            extra_arrays={
                "lag_coefficients": lag_coefficients,
                "lag_function_codes": lag_function_codes,
            },
            metadata={
                "dataset_id": dataset_id,
                "dataset": DATASET_KEY,
                "generator": "generate_data_mod.py",
                "generator_source": "https://github.com/EyringMLClimateGroup/debeire24clear_Bagged_TimeSeries_Causality.git",
                "upstream_commit": UPSTREAM_REVISION.commit,
                "regime": regime["regime"],
                "seed": requested_seed,
                "accepted_model_seed": model_seed,
                "stationarity_retries": retry,
                "n_variables": n_variables,
                "n_samples": n_samples,
                "fit_order": FIT_ORDER,
                "cross_link_count": n_cross_links,
                "cross_link_factor": regime.get("cross_link_factor", 1.5),
                "maximum_autoregressive_coefficient": regime["max_auto"],
                "available_coupling_coefficients": couplings,
                "available_link_functions": list(regime["functions"]),
                "available_noise_types": list(regime["noise_types"]),
                "noise": "+".join(regime["noise_types"]),
                "assigned_noise_types": assigned_noises,
                "assigned_noise_scales": noise_scales,
                "source_paper_contemporaneous_fraction": 0.30,
                "used_contemporaneous_fraction": 0.0,
                "adaptation": (
                    "Contemporaneous links are disabled so every method is "
                    "evaluated on the common positive-lag estimand."
                ),
                "series_layout": "variable, time",
                "truth_layout": "lag, source, target",
                "coefficient_layout": "lag, source, target",
                "function_code_map": {"0": "identity", "1": "quadratic-taper"},
                "truth_scope": "exact positive-lag graph",
            },
        )
        created += int(was_created)
    if stop_requested:
        break
print(
    f"Created {created} new autocorrelated-SCM bundles. "
    "Rerun to verify existing bundles and resume."
)